# 以终为始搞懂旋转位置编码
作者：Double童发发

B站主页：https://space.bilibili.com/323109608

旋转位置编码的理解通常有两种切入点：
1. 从旋转矩阵的定义出发，推导出旋转位置编码的公式，进而用代码实现。
2. 从复数的角度出发，推导出旋转位置编码的公式，进而用代码实现。

这两种确实都是理解旋转位置编码的正确方式，然而门槛太高难以掌握，也不利于初学者理解。今天我们换一个角度，**“以终为始”，直接以Qwen2.5的代码入手，从实现逐步引入旋转位置编码的数学理论**。最终你会发现，从代码入手反而简单，厉害有效的东西往往在实现过程中是足够优雅的。

- 备注：为了便于讲解、测试与复现，会修改部分Qwen2.5的代码，只看最相关的部分
- 参考文件：HuggingFace中`modeling_qwen2.py`文件

## 1. 环境准备
首先需要安装必要的依赖库，可以通过`pip install`安装，主要包括`PyTorch`、`Numpy`、`Transformers`库，如果已经安装可以跳过这一步。

## 2. 必要库导入

In [17]:
import torch
import torch.nn as nn
import transformers
from typing import Tuple, Optional, Callable

## 3. 旋转位置编码的代码实现过程
### 3.1. Qwen2RotaryEmbedding类
首次看代码一定是找到和旋转位置几个字最相关的类去看，那便是`modeling_qwen2.py`文件中的`Qwen2RotaryEmbedding`类，咱们首先看看这个类的代码：

In [24]:
class Qwen2RotaryEmbedding(nn.Module):
    """Qwen2RotaryEmbedding类，基于modeling_qwen2.py中的Qwen2RotaryEmbedding类修改得到
    
    Args:
        head_dim (int): 每个注意力头的维度
        base (int, optional): 旋转位置编码中基频率，默认为10000
        device (torch.device, optional): 设备，默认为None

    """

    def __init__(self,
                 head_dim: int,
                 base: Optional[int] = 10000,
                 device: Optional[torch.device] = None):
        super().__init__()

        # Compute the inverse frequencies
        inv_freq = 1.0 / (base**(torch.arange(
            0, head_dim, 2, dtype=torch.int64).float().to(device) / head_dim)
                          )  # [head_dim//2]
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    @torch.no_grad()
    def forward(self, x, position_ids):
        # Core RoPE block
        # x：[batch_size, seq_len, head_dim]
        # position_ids：[batch_size, seq_len]
        # inv_freq：[head_dim//2] -> inv_freq：[batch_size, head_dim//2, 1]
        inv_freq_expanded = self.inv_freq[None, :, None].float().expand(
            position_ids.shape[0], -1, 1)
        # position_ids_expanded维度：[batch_size, 1, seq_len]
        position_ids_expanded = position_ids[:, None, :].float()
        # 确定张量所在的设备
        device_type = x.device.type
        device_type = device_type if isinstance(
            device_type, str) and device_type != "mps" else "cpu"

        # autocast自动选择合适的精度
        with torch.autocast(device_type=device_type, enabled=False):
            # freqs维度：[batch_size, head_dim//2, seq_len]
            freqs = (inv_freq_expanded.float()
                     @ position_ids_expanded.float()).transpose(1, 2)
            # emb维度：[batch_size, head_dim//2, seq_len, 2]
            emb = torch.cat((freqs, freqs), dim=-1)
            # cos维度：[batch_size, head_dim//2, seq_len, 2]
            cos = emb.cos()
            # sin维度：[batch_size, head_dim//2, seq_len, 2]
            sin = emb.sin()

        return cos.to(dtype=x.dtype), sin.to(dtype=x.dtype)

#### 测试一下

In [30]:
# 所需变量声明
head_dim = 8
base = 10000
batch_size = 4
position_ids = torch.arange(6, device=torch.device("cuda" if torch.cuda.is_available() else "cpu")).repeat(batch_size, 1)   # [batch_size, seq_len]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(batch_size, 6, head_dim, device=device) # [batch_size, seq_len, head_dim]

# 初始化Qwen2RotaryEmbedding类
rotary_embedding = Qwen2RotaryEmbedding(head_dim, base, device)
cos, sin = rotary_embedding(x, position_ids)

print(cos[0])
print(cos[1])
print(sin[0])
print(cos.shape)
# print(sin)
print(sin.shape)

tensor([[ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.9950,  0.9999,  1.0000,  0.5403,  0.9950,  0.9999,  1.0000],
        [-0.4161,  0.9801,  0.9998,  1.0000, -0.4161,  0.9801,  0.9998,  1.0000],
        [-0.9900,  0.9553,  0.9996,  1.0000, -0.9900,  0.9553,  0.9996,  1.0000],
        [-0.6536,  0.9211,  0.9992,  1.0000, -0.6536,  0.9211,  0.9992,  1.0000],
        [ 0.2837,  0.8776,  0.9988,  1.0000,  0.2837,  0.8776,  0.9988,  1.0000]],
       device='cuda:0')
tensor([[ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.9950,  0.9999,  1.0000,  0.5403,  0.9950,  0.9999,  1.0000],
        [-0.4161,  0.9801,  0.9998,  1.0000, -0.4161,  0.9801,  0.9998,  1.0000],
        [-0.9900,  0.9553,  0.9996,  1.0000, -0.9900,  0.9553,  0.9996,  1.0000],
        [-0.6536,  0.9211,  0.9992,  1.0000, -0.6536,  0.9211,  0.9992,  1.0000],
        [ 0.2837,  0.8776,  0.9988,  1.0000,  0.2837,  0.8776,  0.9988,  

In [5]:
class Qwen2Attention(nn.Module):
    """
    Multi-headed attention from 'Attention Is All You Need' paper
    基于Qwen2.5Attention类简化修改
    """
    
    """Qwen2Attention
    
    Args:
        head_dim (int): 每个注意力头维度
        num_attention_heads (int): 注意力头数量
        attention_dropout (float, optional): 注意力dropout概率
    """

    def __init__(self, 
                 head_dim: int,
                 num_attention_heads: int,
                 attention_dropout: float = 0.0,
                 ):
        super().__init__()
        self.head_dim = head_dim # 注意力头维度
        self.scaling = self.head_dim**-0.5 # 缩放因子
        self.attention_dropout = attention_dropout # 注意力dropout
        self.hidden_size = num_attention_heads * head_dim # 隐藏层维度
        self.is_causal = True # 是否为因果关系
        # Q、K、V、O的线性层
        self.q_proj = nn.Linear(self.hidden_size, num_attention_heads * self.head_dim, bias=True)
        self.k_proj = nn.Linear(self.hidden_size, num_attention_heads * self.head_dim, bias=True)
        self.v_proj = nn.Linear(self.hidden_size, num_attention_heads * self.head_dim, bias=True)
        self.o_proj = nn.Linear(num_attention_heads * self.head_dim, self.hidden_size, bias=False)
    
    def apply_rotary_pos_emb(self, q, k, cos, sin, position_ids=None, unsqueeze_dim=1):
        """Applies Rotary Position Embedding to the query and key tensors.
        该函数来自Modeling_qwen2.py，进行了修改简化
  
        Args:
            q (`torch.Tensor`): The query tensor.
            k (`torch.Tensor`): The key tensor.
            cos (`torch.Tensor`): The cosine part of the rotary embedding.
            sin (`torch.Tensor`): The sine part of the rotary embedding.
            position_ids (`torch.Tensor`, *optional*):
                Deprecated and unused.
            unsqueeze_dim (`int`, *optional*, defaults to 1):
                The 'unsqueeze_dim' argument specifies the dimension along which to unsqueeze cos[position_ids] and
                sin[position_ids] so that they can be properly broadcasted to the dimensions of q and k. For example, note
                that cos[position_ids] and sin[position_ids] have the shape [batch_size, seq_len, head_dim]. Then, if q and
                k have the shape [batch_size, heads, seq_len, head_dim], then setting unsqueeze_dim=1 makes
                cos[position_ids] and sin[position_ids] broadcastable to the shapes of q and k. Similarly, if q and k have
                the shape [batch_size, seq_len, heads, head_dim], then set unsqueeze_dim=2.
        Returns:
            `tuple(torch.Tensor)` comprising of the query and key tensors rotated using the Rotary Position Embedding.
        """
        cos = cos.unsqueeze(unsqueeze_dim)
        sin = sin.unsqueeze(unsqueeze_dim)
        q_embed = (q * cos) + (self.rotate_half(q) * sin)
        k_embed = (k * cos) + (self.rotate_half(k) * sin)
        return q_embed, k_embed

    def rotate_half(self, x):
        """Rotates half the hidden dims of the input."""
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)
    
    def attention_forward(
        self,
        module: nn.Module,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        scaling: float,
        attention_mask: Optional[torch.Tensor] = None,
        dropout: Optional[float] = 0.0,
):
        """attention_forward：修改自Modeling_qwen2.py中的eager_attention_forward函数

        Args:
            module (nn.Module): PyTorch用于计算注意力机制的类
            query (torch.Tensor): 查询张量
            key (torch.Tensor): 键张量
            value (torch.Tensor): 值张量
            scaling (float): 缩放因子
            attention_mask (Optional[torch.Tensor]): 注意力掩码
            dropout (float, optional): 注意力dropout概率

        Returns:
            Tuple[torch.Tensor, torch.Tensor]: 注意力机制计算完的输出与注意力权重
        """
        key_states = key  # [batch_size, num_attention_heads, seq_len, head_dim]
        value_states = value  # [batch_size, num_attention_heads, seq_len, head_dim]

        attn_weights = torch.matmul(query, key_states.transpose(2, 3)) * scaling # [batch_size, num_attention_heads, seq_len, seq_len]
        
        # 暂不考虑mask
        if attention_mask is not None:
            causal_mask = attention_mask[:, :, :, : key_states.shape[-2]]
            attn_weights = attn_weights + causal_mask

        # 计算注意力权重
        attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query.dtype)
        # 计算注意力权重dropout
        attn_weights = nn.functional.dropout(attn_weights, p=dropout, training=module.training)
        
        # 计算注意力输出
        attn_output = torch.matmul(attn_weights, value_states) # [batch_size, num_attention_heads, seq_len, head_dim]
        attn_output = attn_output.transpose(1, 2).contiguous() # [batch_size, seq_len, num_attention_heads, head_dim]

        return attn_output, attn_weights

    def forward(
        self,
        hidden_states: torch.Tensor,
        position_embeddings: Tuple[torch.Tensor, torch.Tensor],
        attention_mask: Optional[torch.Tensor],
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[Tuple[torch.Tensor]]]:
        # hidden_states: [batch_size, seq_len, hidden_size]
        # position_embeddings: [batch_size, seq_len, head_dim]
        # attention_mask: [batch_size, seq_len]  随机生成，并不主要考虑
        
        
        input_shape = hidden_states.shape[:-1]  # [batch_size, seq_len]
        hidden_shape = (*input_shape, -1, self.head_dim) # [batch_size, seq_len, num_attention_heads, head_dim]  

        query_states = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2) # [batch_size, num_attention_heads, seq_len, head_dim]
        key_states = self.k_proj(hidden_states).view(hidden_shape).transpose(1, 2) # [batch_size, num_attention_heads, seq_len, head_dim]
        value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2) # [batch_size, num_attention_heads, seq_len, head_dim]

        # 关键！来自于position_embeddings，我们需要想办法获取
        cos, sin = position_embeddings
        query_states, key_states = self.apply_rotary_pos_emb(query_states, key_states, cos, sin)


        attention_interface: Callable = self.attention_forward
        
        attn_output, attn_weights = attention_interface(
            self,
            query_states,
            key_states,
            value_states,
            attention_mask,
            dropout=0.0 if not self.training else self.attention_dropout,
            scaling=self.scaling,
        )

        attn_output = attn_output.reshape(*input_shape, -1).contiguous()
        attn_output = self.o_proj(attn_output)
        return attn_output, attn_weights


#### 测试一下

In [ ]:
# 为了测试我们先随便搞一个PyTorch的类
class TestAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x
module = TestAttention()
# q, k, v的shape为[batch_size, num_attention_heads, seq_len, head_dim]
q = torch.randn(1, 4, 8, 32)
k = torch.randn(1, 4, 8, 32)
v = torch.randn(1, 4, 8, 32)

attn_output, attn_weights = attention_forward(module=module, 
                                              query=q, 
                                              key=k, 
                                              value=v, 
                                              scaling=1.0, 
                                              attention_mask=None, 
                                              dropout=0.0)
print(attn_output.shape)
print(attn_weights.shape)

## 